In [ ]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import shapely
from google.cloud import secretmanager

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_feed_entries,
    get_schedule_rt_stop_times,
    get_schedule_rt_headways
)
from src.metric_calculation import calculate_travel_time_variability, calculate_expected_wait, calculate_on_time_performance

## Configuration

In [ ]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

In [ ]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

# early/late definition for OTP
EARLY_SECONDS = 60
LATE_SECONDS = 5 * 60

## Data/Calculation

In [ ]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))

# carto basemap tiles, authenticated with the api key held in GCP Secret Manager
CARTO_SECRET_PROJECT = os.environ.get("CARTO_SECRET_PROJECT", "cal-itp-data-infra")
CARTO_BASEMAPS_KEY = secretmanager.SecretManagerServiceClient().access_secret_version(
    name=f"projects/{CARTO_SECRET_PROJECT}/secrets/CARTO_BASEMAPS_KEY/versions/latest"
).payload.data.decode("utf-8").strip()
CARTO_TILES = f"https://{{s}}.basemaps.cartocdn.com/light_all/{{z}}/{{x}}/{{y}}{{r}}.png?key={CARTO_BASEMAPS_KEY}"
CARTO_ATTRIBUTION = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

In [ ]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

In [ ]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

In [ ]:
# Calculate travel time variability
schedule_rt_stop_times = get_schedule_rt_stop_times(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)

output_time_difference_variance = calculate_travel_time_variability(
    schedule_rt_stop_times
)

# Calculate schedule performance
on_time = calculate_on_time_performance(
    schedule_rt_stop_times,
    early_seconds=EARLY_SECONDS,
    late_seconds=LATE_SECONDS,
)

# Calculate headway performance
schedule_rt_headways = get_schedule_rt_headways(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
headway_summary = calculate_expected_wait(schedule_rt_headways)


## Results

### Travel Time Variability

#### Definition

The 80th/20th Percentile ratio of durations between specified points on a vehicle’s trip. Reported for the start of the TSP corridor to the end of the TSP corridor and between each Timepoint along the TSP corridor.

#### Reporting Grain

Per Timepoint, Service Pattern, Day Part, and Reporting Period (currently ad hoc for a week).

In [ ]:
# Time variance metric per segment for northbound trip
output_time_difference_variance.loc[
    (output_time_difference_variance.feed_key == "d71492afe126cd61f629b17c3c056fcd")
    & (output_time_difference_variance.shape_id == "ECR2069")
    & (output_time_difference_variance.stop_sequence != 1)
].head(15)

In [ ]:
# generate map
MAP_TIME_PERIOD = "Midday"

# TODO - restrict this map to just shape id = ECR2069, feed_key = d71492afe126cd61f629b17c3c056fcd

timepoints = schedule_rt_stop_times[
    ["shape_id", "feed_key", "stop_sequence", "stop_id", "pt_geom"]
].drop_duplicates()
timepoints_gdf = gpd.GeoDataFrame(
    timepoints.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(timepoints["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

segments_by_shape = []
for shape_id, feed_key, shape_geom in matching_shapes[
    ["shape_id", "feed_key", "geometry"]
].itertuples(index=False):
    timepoints_for_shape = timepoints_gdf.loc[
        (timepoints_gdf.shape_id == shape_id) & (timepoints_gdf.feed_key == feed_key)
    ].copy()
    timepoints_for_shape["projected_distance"] = timepoints_for_shape.geometry.map(
        lambda point_geom: shape_geom.project(point_geom)
    )
    timepoints_for_shape["prior_projected_distance"] = timepoints_for_shape[
        "projected_distance"
    ].shift()
    timepoints_for_shape["segment_geometry"] = timepoints_for_shape[
        ["projected_distance", "prior_projected_distance"]
    ].apply(
        lambda row: (
            shapely.ops.substring(
                shape_geom, row["projected_distance"], row["prior_projected_distance"]
            )
            if not row.isna().any()
            else None
        ),
        axis=1,
    )
    segments_by_shape.append(
        timepoints_for_shape.set_geometry("segment_geometry")[
            ["shape_id", "feed_key", "stop_sequence", "stop_id", "segment_geometry"]
        ]
    )

# the leading timepoint of each pattern has no preceding stop, so no segment
segments_gdf = pd.concat(segments_by_shape).dropna(subset=["segment_geometry"])
segments_gdf = gpd.GeoDataFrame(
    segments_gdf, geometry="segment_geometry"
).set_crs(4326)

segments_for_period = segments_gdf.merge(
    output_time_difference_variance.loc[
        output_time_difference_variance["time_period"] == MAP_TIME_PERIOD
    ],
    on=["shape_id", "feed_key", "stop_sequence"],
    how="inner",
)
segments_for_period = segments_for_period.loc[
    (segments_for_period.stop_sequence != 2) #TODO: there seems to be an issue where this segment overlaps the entire route, so I've manually removed it
    & (segments_for_period.shape_id == "ECR2069")
]

# the ratio and the two percentiles behind it, so a segment can be read as more
# than a colour
for column in [
    "travel_time_variability",
    "p80_travel_time_seconds",
    "p20_travel_time_seconds",
]:
    segments_for_period[column] = segments_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
travel_time_variability_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles=CARTO_TILES,
    attr=CARTO_ATTRIBUTION,
    name="service patterns",
)
segments_for_period.explore(
    m=travel_time_variability_map,
    column="travel_time_variability",
    cmap="plasma",
    style_kwds={"weight": 6, "opacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "n_rt_trips",
        "p80_travel_time_seconds",
        "p20_travel_time_seconds",
        "travel_time_variability",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Trips",
            "80th pct run time (s)",
            "20th pct run time (s)",
            "p80/p20 ratio",
        ]
    },
    legend_kwds={
        "caption": f"p80/p20 ratio of {MAP_TIME_PERIOD.lower()} run times into each timepoint"
    },
    name=f"{MAP_TIME_PERIOD} trip duration variability",
)
travel_time_variability_map.save(f"trip_duration_variability_{MAP_TIME_PERIOD.lower()}.html")

In [ ]:
# Map of midday day part, also northbound only
travel_time_variability_map

### Expected Wait

#### Definition
The proportion of the time during the Day Part in which an arriving passenger would experience an Actual Headway that is greater than the Scheduled Headway.

#### Reporting Grain
Service Pattern, stop, Day Part, and Reporting Period

In [ ]:
headway_summary.loc[(headway_summary.time_period == "Midday") & (headway_summary.stop_sequence != 1)]

In [ ]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by how much of the day part would have left a rider waiting
# longer than the schedule promised.
MAP_TIME_PERIOD = "Midday"

# The headways query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_headways[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    headway_summary.loc[headway_summary["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# percent_time_excess_wait is a proportion; the map reads better in percent. The
# colour and the tooltip use the one column, so the legend matches what is hovered
stops_for_period["percent_time_excess_wait"] = (
    stops_for_period["percent_time_excess_wait"] * 100
).round(1)
for column in [
    "scheduled_expected_wait_seconds",
    "rt_expected_wait_seconds",
    "excess_wait_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
headway_performance_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles=CARTO_TILES,
    attr=CARTO_ATTRIBUTION,
    name="service patterns",
)
stops_for_period.explore(
    m=headway_performance_map,
    column="percent_time_excess_wait",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "scheduled_expected_wait_seconds",
        "rt_expected_wait_seconds",
        "excess_wait_seconds",
        "percent_time_excess_wait",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Scheduled expected wait (s)",
            "Actual expected wait (s)",
            "Excess wait (s)",
            "% of time waiting over schedule",
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} spent waiting longer than scheduled"
    },
    name=f"{MAP_TIME_PERIOD} excess wait",
)
headway_performance_map.save(f"expected_wait_{MAP_TIME_PERIOD.lower()}.html")

In [ ]:
# Map of all stops for one day part

headway_performance_map

### Schedule Adherence at Timepoint

#### Definition
At each transit stop, the proportion of trips that are less than five minutes late and less than one minute early.

#### Reporting Grain
Reporting Period and Day Part. (also mapped at timepoints here)

### Average Deviation from Schedule

#### Definition

The difference between when the bus is scheduled to arrive at each stop and when the bus arrives based on real time data. Reported as 50th and 80th percentile.



In [ ]:
# OTP by stop for the northbound direction midday only

on_time.by_period

In [ ]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by the share of arrivals that were not on time.
MAP_TIME_PERIOD = "Midday"

# The stop times query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_stop_times[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    on_time.by_stop.loc[on_time.by_stop["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# the proportions are 0 to 1; the map reads better in percent. The colour and the
# tooltip use the one column, so the legend matches what is hovered
for column in ["proportion_early", "proportion_late", "proportion_not_on_time", "schedule_adherence"]:
    stops_for_period[column] = (stops_for_period[column] * 100).round(1)
for column in [
    "p50_absolute_deviation_seconds",
    "p80_absolute_deviation_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
otp_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles=CARTO_TILES,
    attr=CARTO_ATTRIBUTION,
    name="service patterns",
)
stops_for_period.explore(
    m=otp_map,
    column="schedule_adherence",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "n_timepoint_arrivals",
        "p50_absolute_deviation_seconds",
        "p80_absolute_deviation_seconds",
        "proportion_early",
        "proportion_late",
        "proportion_not_on_time",
        "schedule_adherence"
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Arrivals",
            "Median deviation (s)",
            "80th pct deviation (s)",
            "% early",
            "% late",
            "% not on time",
            "On Time Performance"
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} arrivals on time"
    },
    name=f"{MAP_TIME_PERIOD} on time performance",
)
otp_map.save(f"on_time_performance_{MAP_TIME_PERIOD.lower()}.html")


In [ ]:
otp_map